# 08 — Tools

@tool/defineTool, auto-injected transfer_call/end_call, dynamic variables, custom tools, schema validation.


## Optional: run in Docker

Uncomment the cell below to launch JupyterLab + EmbeddedServer in a container. It's idempotent and a no-op once you're already inside the container. See `examples/notebooks/python/Dockerfile` and `docker-compose.yml` for what it builds.


In [ ]:
# Optional — launch the patter-notebooks Docker stack from this cell.
# Skip this cell to run on your host Python. Idempotent if uncommented.
#
# import _setup
# _setup.start_docker()             # build + up -d, prints http://localhost:8888
# _setup.start_docker(open_url=True)  # …and open the browser tab


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
%load_ext autoreload
%autoreload 2

import _setup
env = _setup.load()
print(f'getpatter version target: {env.patter_version}')


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


In [ ]:
import sys
import getpatter
with _setup.cell('version_check', tier=1, env=env) as ok:
    if ok:
        print(f'getpatter {getpatter.__version__} on Python {sys.version.split()[0]}')
        assert getpatter.__version__ >= env.patter_version, \
            f'installed {getpatter.__version__} < target {env.patter_version}'


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


In [ ]:
from getpatter import Patter, Twilio
with _setup.cell('local_mode', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(
                account_sid='ACtest00000000000000000000000000',
                auth_token='test',
            ),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        # Verify carrier is wired up via LocalConfig.
        assert p._local_config.telephony_provider == 'twilio'
        assert p._local_config.phone_number == '+15555550100'
        print(f'provider={p._local_config.telephony_provider}  phone={p._local_config.phone_number}')


### Cloud mode (coming soon)
When `api_key=` is supported, Patter cloud handles telephony. For now, the SDK raises `NotImplementedError` — this cell verifies the guard.


In [ ]:
from getpatter import Patter
with _setup.cell('cloud_mode', tier=1, env=env) as ok:
    if ok:
        try:
            Patter(api_key='pt_test_xxx')
            raise AssertionError('expected NotImplementedError — cloud mode guard missing')
        except NotImplementedError as exc:
            print(f'cloud mode guard OK: {exc}')


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the engine from `engine=` / `stt=`/`tts=`.


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI
with _setup.cell('agent_engines', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid='ACtest00000000000000000000000000',
                           auth_token='test'),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        rt = p.agent(system_prompt='hi', engine=OpenAIRealtime(api_key='sk-test'))
        cv = p.agent(system_prompt='hi', engine=ElevenLabsConvAI(api_key='el-test', agent_id='a1'))
        pl = p.agent(system_prompt='hi')  # default: OpenAI Realtime fallback
        assert rt.provider == 'openai_realtime', rt.provider
        assert cv.provider == 'elevenlabs_convai', cv.provider
        assert pl.provider == 'openai_realtime', pl.provider
        print(f'rt.provider={rt.provider}  cv.provider={cv.provider}  pl.provider={pl.provider}')


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


### Cloud mode
Same SDK, just an `api_key=` instead of a carrier — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the mode from `engine=` / `stt=`/`tts=`.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises the `@tool` decorator, `Tool()` factory, and agent tool registration.


### `tool()` factory


In [ ]:
from getpatter import tool
with _setup.cell('tool_decorator', tier=1, env=env) as ok:
    if ok:
        @tool
        def get_weather(city: str, units: str = 'celsius') -> str:
            """Get the current weather for a city."""
            return f'Sunny, 22{units[0].upper()} in {city}'

        print(f'name:        {get_weather.name}')
        print(f'description: {get_weather.description}')
        print(f'call:        {get_weather.handler(city="Paris")}')
        assert get_weather.name == 'get_weather'
        assert 'city' in get_weather.description or get_weather.handler is not None


### `Tool()` constructor


In [ ]:
from getpatter import Tool
with _setup.cell('tool_inline', tier=1, env=env) as ok:
    if ok:
        search_tool = Tool(
            name='web_search',
            description='Search the web for up-to-date information.',
            parameters={
                'query': {'type': 'string', 'description': 'The search query'},
                'num_results': {'type': 'integer', 'default': 5},
            },
            handler=lambda query, num_results=5: f'Top {num_results} results for {query!r}',
        )
        print(f'name:        {search_tool.name}')
        print(f'description: {search_tool.description}')
        print(f'call:        {search_tool.handler(query="Patter SDK", num_results=3)}')
        assert search_tool.name == 'web_search'


### Agent with tools list


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, tool, Tool
with _setup.cell('tool_in_agent', tier=1, env=env) as ok:
    if ok:
        @tool
        def book_appointment(date: str, time: str) -> str:
            """Book an appointment for the caller."""
            return f'Appointment booked for {date} at {time}'

        cancel_tool = Tool(
            name='cancel_appointment',
            description='Cancel an existing appointment.',
            parameters={'confirmation_number': {'type': 'string'}},
            handler=lambda confirmation_number: f'Cancelled {confirmation_number}',
        )

        p = Patter(
            carrier=Twilio(account_sid='ACtest00000000000000000000000000', auth_token='test'),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        agent = p.agent(
            system_prompt='You are a helpful booking assistant.',
            engine=OpenAIRealtime(api_key='sk-test'),
            tools=[book_appointment, cancel_tool],
        )
        print(f'Agent tools: {[t.name for t in agent.tools]}')
        assert len(agent.tools) == 2
        assert agent.tools[0].name == 'book_appointment'
        assert agent.tools[1].name == 'cancel_appointment'


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Fires a real tool call mid-call and demonstrates `transfer_call`. Requires `ENABLE_LIVE_CALLS=1`.


### Pre-flight checklist


In [ ]:
with _setup.cell('live_preflight', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env=env) as ok:
    if ok:
        print(f'  carrier:  Twilio {env.twilio_number}  →  {env.target_number}')
        print(f'  tools:    get_time (custom) + transfer_call (auto-injected)')
        print(f'  webhook:  {env.public_webhook_url or "(ngrok auto-launch)"}')


### Live call with custom tool *(T4)*


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, tool
with _setup.cell('live_tools_call', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env=env) as ok:
    if ok:
        @tool
        def lookup_order(order_id: str) -> str:
            """Look up the status of an order by ID."""
            return f'Order {order_id} is shipped — expected delivery: tomorrow.'

        p = Patter(
            carrier=Twilio(account_sid=env.twilio_sid, auth_token=env.twilio_token),
            phone_number=env.twilio_number,
            webhook_url=env.public_webhook_url,
        )
        agent = p.agent(
            system_prompt='You are a demo order assistant. If asked, look up order 12345.',
            engine=OpenAIRealtime(api_key=env.openai_key),
            tools=[lookup_order],
        )
        try:
            await p.call(env.target_number, agent=agent,
                         first_message='Hi! Ask me about order 12345.',
                         ring_timeout=env.max_call_seconds)
            print('✓ Tools call completed')
        finally:
            _setup.hangup_leftover_calls(env)
